# Segmento 3 — Estrarre dati strutturati con un LLM

Abbiamo visto che le regex falliscono con le stringhe di ingredienti reali. E se invece di scrivere più regole, *chiedessimo* semplicemente a un modello linguistico di analizzarle?

In [8]:
from dotenv import load_dotenv
from openai import OpenAI
from datasets import load_dataset
import ast, json, os
import pandas as pd

load_dotenv()
client = OpenAI()
MODEL = "gpt-5.4-nano"

ds = load_dataset("Hieu-Pham/kaggle_food_recipes", split="train")
df = ds.to_pandas().drop(columns=["Unnamed: 0", "Image_Name"])
print(f"{len(df)} recipes loaded. We will use model: {MODEL}")

13501 recipes loaded. We will use model: gpt-5.4-nano


## Chiediamo e basta

La stringa di ingredienti che ha fatto fallire la nostra regex: `"Pinch of crushed red pepper flakes"`. Inviamola a un LLM e vediamo cosa restituisce.

In [10]:
response = client.chat.completions.create(
    model=MODEL,
    messages=[
        {"role": "system", "content": "You parse ingredient strings into structured JSON with keys: quantity, unit, ingredient, notes. Return only JSON."},
        {"role": "user", "content": "Pinch of crushed red pepper flakes"},
    ],
    response_format={"type": "json_object"},
)

result = json.loads(response.choices[0].message.content)
print(json.dumps(result, indent=2))

{
  "quantity": 1,
  "unit": "pinch",
  "ingredient": "crushed red pepper flakes",
  "notes": "crushed"
}


## Una funzione di estrazione riutilizzabile

Incapsuliamo tutto in una funzione che possiamo chiamare su qualsiasi stringa di ingredienti.

In [11]:
SYSTEM_PROMPT = """You parse ingredient strings into structured JSON.
Return a JSON object with these keys:
- quantity: the numeric amount (use decimals for fractions, e.g. 2.75). Use "to taste" or "as needed" when appropriate. Null if missing.
- unit: the measurement unit (e.g. tsp, Tbsp, cup, lb, oz). Null if missing.
- ingredient: the ingredient name, cleaned up.
- notes: any preparation or extra info (e.g. "divided", "melted", "for frying"). Null if none.
Return ONLY the JSON object, nothing else."""

def parse_ingredient_llm(text):
    """Parse a single ingredient string using the LLM."""
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": text},
        ],
        response_format={"type": "json_object"},
    )
    return json.loads(response.choices[0].message.content)

# Quick test
parse_ingredient_llm("2¾ tsp. kosher salt, divided, plus more")

{'quantity': 2.75,
 'unit': 'tsp',
 'ingredient': 'kosher salt',
 'notes': 'divided, plus more'}

## La rivincita — ogni stringa che ha fatto fallire la nostra regex

Questi sono esattamente gli stessi 7 esempi del segmento 2 in cui la regex ha fallito. Vediamo come se la cava l'LLM.

In [12]:
hard_examples = [
    "Pinch of crushed red pepper flakes",
    "1 (3½–4-lb.) whole chicken",
    "One 14-ounce can whole peeled tomatoes",
    "Salt and pepper",
    "Vegetable oil, for frying",
    "Juice of 2 lemons",
    "6 Tbsp. unsalted butter, melted, plus 3 Tbsp. room temperature",
]

results = []
for ing in hard_examples:
    parsed = parse_ingredient_llm(ing)
    results.append({"raw": ing, **parsed})
    print(f"'{ing}'")
    print(f"  → {parsed}\n")

df_parsed = pd.DataFrame(results)
df_parsed

'Pinch of crushed red pepper flakes'
  → {'quantity': None, 'unit': None, 'ingredient': 'crushed red pepper flakes', 'notes': 'pinch'}

'1 (3½–4-lb.) whole chicken'
  → {'quantity': 1, 'unit': None, 'ingredient': 'whole chicken', 'notes': '3.5–4-lb'}

'One 14-ounce can whole peeled tomatoes'
  → {'quantity': 14, 'unit': 'oz', 'ingredient': 'whole peeled tomatoes', 'notes': None}

'Salt and pepper'
  → {'quantity': None, 'unit': None, 'ingredient': 'salt and pepper', 'notes': None}

'Vegetable oil, for frying'
  → {'quantity': None, 'unit': None, 'ingredient': 'vegetable oil', 'notes': 'for frying'}

'Juice of 2 lemons'
  → {'quantity': 2, 'unit': None, 'ingredient': 'lemons', 'notes': 'juice'}

'6 Tbsp. unsalted butter, melted, plus 3 Tbsp. room temperature'
  → {'quantity': 6, 'unit': 'Tbsp', 'ingredient': 'unsalted butter', 'notes': 'melted, plus 3 Tbsp room temperature'}



,raw,quantity,unit,ingredient,notes
0,Pinch of crushed red pepper flakes,NaN,None,crushed red pepper flakes,pinch
1,1 (3½–4-lb.) whole chicken,1.0,None,whole chicken,3.5–4-lb
2,One 14-ounce can whole peeled tomatoes,14.0,oz,whole peeled tomatoes,None
3,Salt and pepper,NaN,None,salt and pepper,None
4,"Vegetable oil, for frying",NaN,None,vegetable oil,for frying
5,Juice of 2 lemons,2.0,None,lemons,juice
6,"6 Tbsp. unsalted butter, melted, plus 3 Tbsp. ...",6.0,Tbsp,unsalted butter,"melted, plus 3 Tbsp room temperature"


## Parsing di una ricetta completa

Prendiamo il Miso-Butter Roast Chicken (22 ingredienti) e passiamo ogni ingrediente attraverso l'LLM.

In [ ]:
recipe = df.iloc[0]
print(f"Recipe: {recipe['Title']}\n")

ingredients = ast.literal_eval(recipe["Ingredients"])
parsed = [parse_ingredient_llm(ing) for ing in ingredients]
df_recipe = pd.DataFrame(parsed)
df_recipe

## Oltre gli ingredienti

L'LLM non si limita ad analizzare stringhe — comprende il testo. Chiediamogli di estrarre metadati di alto livello da una ricetta completa: cucina, difficoltà, tecniche, informazioni dietetiche, tempi.

In [13]:
METADATA_PROMPT = """Analyze this recipe and return a JSON object with:
- cuisine: the most likely cuisine type (e.g. "Italian", "Japanese", "American", "French")
- difficulty: "easy", "medium", or "hard"
- techniques: list of cooking techniques used (e.g. ["roast", "sauté", "braise"])
- dietary: list of applicable tags (e.g. ["vegetarian", "gluten-free", "dairy-free"]). Empty list if none.
- total_time_minutes: estimated total time including prep and cook time. Null if unclear.
Return ONLY the JSON object."""

def extract_metadata(row):
    """Extract high-level metadata from a full recipe."""
    recipe_text = f"Title: {row['Title']}\nIngredients: {row['Ingredients']}\nInstructions: {row['Instructions']}"
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": METADATA_PROMPT},
            {"role": "user", "content": recipe_text},
        ],
        response_format={"type": "json_object"},
    )
    return json.loads(response.choices[0].message.content)

# Try it on the Miso-Butter Roast Chicken
meta = extract_metadata(df.iloc[0])
print(f"Recipe: {df.iloc[0]['Title']}\n")
print(json.dumps(meta, indent=2))

Recipe: Miso-Butter Roast Chicken With Acorn Squash Panzanella

{
  "cuisine": "American",
  "difficulty": "medium",
  "techniques": [
    "roast",
    "season",
    "rest",
    "toss",
    "bake",
    "saut\u00e9",
    "deglaze",
    "simmer",
    "thicken",
    "whisk"
  ],
  "dietary": [],
  "total_time_minutes": 165
}


## Confronto tra ricette

Eseguiamo l'analisi su alcune ricette diverse — un cocktail, un contorno semplice e una cena complessa — e vediamo i risultati a confronto.

In [ ]:
# Pick a diverse set: cocktail (idx 5), simple side (idx 1), complex dinner (idx 0), breakfast (idx 42)
sample_indices = [5, 1, 0, 42]
rows = []

for idx in sample_indices:
    row = df.iloc[idx]
    print(f"Processing: {row['Title']}...")
    meta = extract_metadata(row)
    meta["title"] = row["Title"]
    rows.append(meta)

df_meta = pd.DataFrame(rows)
df_meta[["title", "cuisine", "difficulty", "techniques", "dietary", "total_time_minutes"]]